# Requirement 6: User authorization

> Authenticate and authorize the requesting user before creating or accessing a session.

**Covered by:** Caller

**How it is fulfilled:** This is the calling application's job and the platform does not provide it. The pattern below shows the orchestrator mapping each session to the user who created it, requiring an approved group, and refusing any other user.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
APPROVED_GROUP = "contoso-interpreter-users"
sessions = {}   # session id -> owning user object id

def create_session(user_oid, user_groups):
    if APPROVED_GROUP not in user_groups:
        raise PermissionError("user is not authorized to create a session")
    session_id = uuid.uuid4().hex[:8]
    sessions[session_id] = user_oid
    return session_id

def access_session(user_oid, session_id):
    if sessions.get(session_id) != user_oid:
        raise PermissionError("session belongs to another user")
    return "access granted"


In [3]:
alice = "00000000-aaaa-0000-0000-000000000001"
bob = "00000000-bbbb-0000-0000-000000000002"
sid = create_session(alice, [APPROVED_GROUP])
check("authorized user creates a session", sid in sessions)
check("owner accesses it", access_session(alice, sid) == "access granted")
for label, call in [("user outside the group creates", lambda: create_session(bob, [])),
                    ("another user accesses it", lambda: access_session(bob, sid))]:
    try:
        call(); ok = False
    except PermissionError:
        ok = True
    check(f"{label} -> denied", ok)

PASS  authorized user creates a session
PASS  owner accesses it
PASS  user outside the group creates -> denied
PASS  another user accesses it -> denied


**Caveat:** This is an illustration of the caller-side pattern. In production the user identity comes from the Entra token presented to the interpreter tool (for example an MCP endpoint in a Foundry Toolbox).